In [298]:
import os
import re
from tqdm import tqdm
from typing import Iterator, Tuple

import cv2
from PIL import Image

import numpy as np
import pandas as pd

import tokenizers
import transformers
from tokenizers import trainers, processors, pre_tokenizers

import torch
import torchvision
import torchsummary
from torch import nn
from torch.utils import data
from torchvision import io, transforms
from torchvision.transforms import functional

In [526]:
#global parameters
D_MODEL = 1024
N_HEADS = 2
NUM_LAYERS = 1

In [527]:
#checking cuda
torch.cuda.is_available()

True

In [528]:
def resnet_trans(tensor_image):
    """
    Parameters
    ----------
    tensor_image: tensor (that represents some kind of an image)
    
    returns
    ----------
    Tensor image transformed for ResNet; with dtype float32
    """
    #change dtype to float
    tensor_image = tensor_image.to(dtype = torch.float32)
    #apply necessary transformations for ResNet
    data_transforms = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    return data_transforms(tensor_image)

In [325]:
def get_tokenizer_object(words_iterable):
    """
    Parameters
    ----------
    Iterable of words

    returns
    -------
    a tokenizer object that implements all the necessary 
    special tokens additions

    Description
    ------------
    Takes in an iterable of words and returns a HF tokenizer object over them
    that appends all the necessary special tokens

    Note
    -----
    The tokenizer would expect the words that have single spaces between the letters
    example: 'cat' --> 'c a t'
    """
    #get maximum length of words
    max_length = 0
    for word in words_iterable:
        max_length = max(max_length,len(word))
    #adding 2 for sos and eos tokens 
    max_length += 2
    #creating words with spaces between letters
    main_iterable_of_words = [' '.join([letter for letter in word]) for word in words_iterable]
    #specifiing tokenizer parameters
    tokenizer_obj = tokenizers.Tokenizer(model=tokenizers.models.WordLevel(unk_token='<unk>'))
    tokenizer_obj.pre_tokenizer = pre_tokenizers.Whitespace()
    tokenizer_obj.post_processor = processors.TemplateProcessing(single="<sos> $A <eos>", special_tokens=[('<sos>',2), ('<eos>',3)])
    #enabling padding with maximun length
    tokenizer_obj.enable_padding(direction='right', pad_id = 0, pad_token='<pad>', length=max_length)
    #training tokenizer object
    tokenizer_trainer = trainers.WordLevelTrainer(vocab_size=100, special_tokens=['<pad>','<unk>','<sos>','<eos>'])
    tokenizer_obj.train_from_iterator(iterator=main_iterable_of_words, trainer=tokenizer_trainer)
    #transformers tokenizer
    transformers_tokenizer = transformers.PreTrainedTokenizerFast(tokenizer_object = tokenizer_obj)
    return transformers_tokenizer 

In [326]:
def get_dataset(path_to_folder = '/home/luchian/all_data/datasets/text_rec_dataset'):
    """
    Parameters
    ----------
    An absolute path to folder with dataset
    
    returns
    --------
    a dataframe that has the column names [image_path, target]
    with corresponding data

    Description
    ------------
    Takes in an absolute path to a folder that contains data in the form
    folder_with_target_name[images]
    i.e. each target has a folder of its name with images in it belonging to that target
    """
    #create a dataframe
    main_frame = pd.DataFrame(columns=['ImagePath','Target'])
    #get target folders
    list_of_target_folders = os.listdir(path_to_folder)
    #for each target folder get the images with that target
    for target_name in list_of_target_folders:
        images = os.listdir(path_to_folder+'/'+target_name)
        pd_frame = pd.DataFrame({'ImagePath':[path_to_folder+'/'+target_name+'/'+image_path for image_path in images], 'Target':[target_name for _ in range(len(images))]})
        main_frame = pd.concat([main_frame, pd_frame], ignore_index=True) #ignore index to make shure we have regular indexing order 
    return main_frame

In [327]:
def make_spaces(word):
    """Makes spaces between the letter of a given word"""
    return ' '.join(list(word))

In [328]:
main_dataset = get_dataset()
main_dataset

,ImagePath,Target
0,/home/luchian/all_data/datasets/text_rec_datas...,dog
1,/home/luchian/all_data/datasets/text_rec_datas...,cat


In [401]:
class TextRecDataset(data.Dataset):
    """Implements dataset for Text Recognition"""
    def __init__(self,dataset: pd.DataFrame, tokenizer: transformers.PreTrainedTokenizerFast, word_processor = make_spaces, picture_processor = resnet_trans):
        self.dataset = dataset
        self.tokenizer = tokenizer
        #processors for words and pictures
        self.word_processor = word_processor
        self.picture_processor = picture_processor

    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self,indx):
        #getting image path and target name
        data_indx_row = self.dataset.iloc[indx,:]
        image_path = data_indx_row['ImagePath']
        target_name = data_indx_row['Target']
        #applying transformations and converting to necessary formats (Tensors)
        tensor_image = self.picture_processor(io.read_image(path = image_path))
        #target names to tensors with a given tokenizer
        target_ids = torch.tensor(self.tokenizer(self.word_processor(target_name)).input_ids, dtype = torch.long)
        return tensor_image, target_ids[:-1], target_ids[1:]

In [515]:
#Enocder-Decoder model for TextRecognition 
class TRModel(nn.Module):
    def __init__(self, vocabulary_size, d_model = D_MODEL, nhead = N_HEADS, num_layers = NUM_LAYERS, padding_indx = 0):
        super().__init__()
        #Encoder
        #load ResNet with default weights
        self.encoder = torchvision.models.resnet18(weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1, progress = True)
        #freeze all layers
        for parameter in self.encoder.parameters():
            parameter.requires_grad = False
        #create and attach new last head
        new_last_head = nn.Linear(512,d_model)
        self.encoder.fc = new_last_head

        #embedding layer
        self.embed_layer = nn.Embedding(num_embeddings=vocabulary_size, embedding_dim=d_model, padding_idx=padding_indx)

        #Decoder
        self.decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead)
        self.decoder = nn.TransformerDecoder(self.decoder_layer, num_layers=num_layers)

        #linear head over vocabulary
        self.linear_head = nn.Linear(d_model,vocabulary_size)

    def forward(self, x_pic_tensor, x_letter_ids):
        """
        parameters
        ----------
        x_pic: tensor of shape batch_size X channels X H x W
        x_text: tensor of shape seq_len X batch_size X d_model (embed_dim)
        """
        encoder_forward = self.encoder(x_pic_tensor).unsqueeze(dim = 0) #transofrmer decoder expects input of shape (seq_len X batch_size X d_model)
        #getting vector embeddings
        letter_embeds = self.embed_layer(x_letter_ids).transpose(0,1) # seq_length has to be first for transformer decoder
        decoder_forward = self.decoder(letter_embeds, memory = encoder_forward)
        clf_head = self.linear_head(decoder_forward)
        return clf_head

In [464]:
TorchDataset = TextRecDataset(main_dataset, get_tokenizer_object(main_dataset['Target']))
TorchDataset

In [440]:
loader = data.DataLoader(dataset = TorchDataset, shuffle = True, batch_size = 2)

In [441]:
sample = next(iter(loader))
print(sample[0].shape)
print(sample[1].shape)

torch.Size([2, 3, 224, 224])
torch.Size([2, 4])


In [442]:
pic_tensor = sample[0]
print(pic_tensor.shape)
letter_ids = sample[1]
print(letter_ids.shape)

torch.Size([2, 3, 224, 224])
torch.Size([2, 4])


In [435]:
pic_tensor = torch.randn(1, 3, 224, 224)
print(pic_tensor.shape)
letter_ids = torch.tensor([[2,3,2,1,3]], dtype = torch.long)
print(letter_ids.shape)

torch.Size([1, 3, 224, 224])
torch.Size([1, 5])


In [529]:
model = TRModel(vocabulary_size=TorchDataset.tokenizer._tokenizer.get_vocab_size())

In [466]:
model(x_pic_tensor = pic_tensor, x_letter_ids = letter_ids).shape

torch.Size([4, 2, 10])

In [485]:
def train_model(model,train_loader,epoch,main_optim,main_loss,print_every = 1,dev = 'cpu'):
    """
    Given training parameters trains the model
    """
    #Handling KeyBoardInterrupt exception error
    try:
        train_losses = []
        #Going through epochs
        for ep in range(epoch):
            model.train()
            #we are going to save the mean of losses
            epoch_losses = []
            for X,y,z in tqdm(train_loader, desc=f'Going through the loader on epoch #{ep+1}'):
                #preparing for forward propogation
                X,y,z = X.to(device = dev), y.to(device = dev), z.to(device = dev)
                main_optim.zero_grad()
                y_pred = model(x_pic_tensor = X, x_letter_ids = y)
                #reshape prediction and true values to go through loss
                y_pred = y_pred.reshape(-1, loader.dataset.tokenizer._tokenizer.get_vocab_size())
                z = z.reshape(-1)
                the_loss = main_loss(y_pred,z)
                #backprop and step
                the_loss.backward()
                main_optim.step()
                #keep track of losses
                epoch_losses.append(the_loss.item())
            #for each epoch we append the mean loss over that epoch
            train_losses.append(round(np.array(epoch_losses).mean().item(),5))
            if ep%print_every == 0:
                print(f'Epoch #{ep+1} | Train loss: {train_losses[-1]}',end = '\n\n')
        return train_losses
    except KeyboardInterrupt:
        return train_losses

In [536]:
model = TRModel(vocabulary_size=TorchDataset.tokenizer._tokenizer.get_vocab_size())

In [537]:
#train
epoch = 2000
batch_size = 2
lr = 2e-10
loader = data.DataLoader(dataset = TorchDataset,shuffle = True,batch_size = batch_size)
optimizer = torch.optim.SGD(model.parameters(),lr = lr,momentum=0.45)
criterion = nn.CrossEntropyLoss(reduction = 'mean')

In [538]:
results = train_model(model = model,
                      train_loader = loader,
                      epoch = epoch,
                      main_optim = optimizer,
                      main_loss = criterion,
                      print_every = 10,
                      dev = 'cpu')

Going through the loader on epoch #1: 100%|██████████| 1/1 [00:00<00:00, 14.75it/s]


Epoch #1 | Train loss: 2.70369



Going through the loader on epoch #11: 100%|██████████| 1/1 [00:00<00:00, 13.47it/s]


Epoch #11 | Train loss: 2.92682



Going through the loader on epoch #21: 100%|██████████| 1/1 [00:00<00:00, 15.20it/s]


...

Going through the loader on epoch #961: 100%|██████████| 1/1 [00:00<00:00, 13.67it/s]


Epoch #961 | Train loss: 2.68007



Going through the loader on epoch #971: 100%|██████████| 1/1 [00:00<00:00, 16.86it/s]


Epoch #971 | Train loss: 2.61821



Going through the loader on epoch #975:   0%|          | 0/1 [00:00<?, ?it/s]


In [ ]:
def model_inference():
    """
    Description
    -----------
    Takes in an image and returns generated text
    """
    pass

(torch.Size([8, 10]), torch.Size([2, 4]))